# NPC Dialogue Engine — End-to-End LoRA Fine-Tune (Colab T4)

This notebook runs the full **train → merge → evaluate** loop for the NPC Dialogue Engine on a free Colab T4 (16GB VRAM).

**What it produces:**
- A LoRA adapter at `models/qwen-npc-lora/final/`
- A merged, deployment-ready model at `models/merged/`
- A second eval report at `results/eval_report_ft.json` for direct comparison against `results/eval_report.json` (the mock baseline)

**Wallclock budget:** ~3-5h on a T4. The training step is the long one (~2-3h on 900 examples × 3 epochs). Eval against the merged model on 20 golden + 15 adversarial prompts is ~5-10 min.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Set the `REPO_BRANCH` cell below if you want a specific commit
3. (Optional) Mount Drive to persist the merged model — see the final cell

**Why this notebook exists:** the checked-in `results/eval_report.json` is a mock-baseline — every metric except latency/safety fails because the mock model can't follow instructions. This notebook produces the real-model evidence that closes the credibility gap.

## 1. Environment check

In [ ]:
!nvidia-smi || echo 'No GPU detected — set Runtime → Change runtime type → T4 GPU'

## 2. Clone the repo

In [ ]:
REPO_URL = 'https://github.com/7ahir/npc-dialogue-engine.git'
REPO_BRANCH = 'main'  # pin to a SHA for reproducibility

import os
if not os.path.isdir('npc-dialogue-engine'):
    !git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL}
%cd npc-dialogue-engine
!git rev-parse --short HEAD

## 3. Install dependencies

Pinning major versions to what the slow tier of the test suite was last verified against. Colab T4 already has CUDA 12.x + a recent torch, so we install on top.

In [ ]:
!pip install -q -e ".[ml,gpu,train]" 2>&1 | tail -5
!pip install -q sse-starlette  # transitive but not always pulled by the editable install

In [ ]:
# Sanity: make sure peft + bitsandbytes import on this CUDA build
import torch, peft, bitsandbytes, transformers, trl
print('torch        :', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('peft         :', peft.__version__)
print('bitsandbytes :', bitsandbytes.__version__)
print('transformers :', transformers.__version__)
print('trl          :', trl.__version__)

## 4. Generate training data

`--examples 30` produces 3 chars × 10 scenarios × 30 = **900 single-turn exchanges**. The model card explains the trade-off: this is a deliberately small, reproducible seed. For a stronger run, swap in a paraphrase-augmentation pass before training (out of scope for the first cycle).

In [ ]:
!python scripts/generate_training_data.py \
    --output data/processed/train.jsonl \
    --examples 30

!wc -l data/processed/train.jsonl
!head -1 data/processed/train.jsonl | python -m json.tool

## 5. Index lore into ChromaDB

The eval pipeline retrieves lore chunks for grounding. If we skip this, the `lore_accuracy` and `grounding_rate` metrics are vacuous.

In [ ]:
!python scripts/index_lore.py 2>&1 | tail -10

## 6. Run LoRA fine-tune

Reads `configs/model_config.yaml` (Qwen 2.5-3B base, r=16 LoRA on q/k/v/o, NF4 4-bit, 3 epochs). Writes the adapter to `models/qwen-npc-lora/final/`.

**This is the long cell.** Expect 2-3h on a T4 with 900 examples × 3 epochs at effective batch size 16.

In [ ]:
import time
_t0 = time.time()
!python src/training/train_lora.py --data-path data/processed/train.jsonl 2>&1 | tail -30
print(f'\nTraining wallclock: {(time.time() - _t0) / 60:.1f} min')

In [ ]:
!ls -la models/qwen-npc-lora/final/

## 7. Merge LoRA adapter into a deployable model

Uses `AutoPeftModelForCausalLM.merge_and_unload()`. The output at `models/merged/` is a plain `transformers` directory — loadable by TGI, vLLM, or llama.cpp converters.

In [ ]:
!python scripts/export_model.py \
    --adapter-path models/qwen-npc-lora/final \
    --output models/merged \
    --dtype bfloat16

!ls -la models/merged/

## 8. Evaluate the fine-tuned model

Same pipeline as the mock baseline (intent → RAG → generation → 7 metrics), but now the generator is the merged Qwen+LoRA. `--model-path` flips the model into transformers mode and points it at the local merged dir.

In [ ]:
!python scripts/run_evaluation.py \
    --model-path models/merged \
    --output results/eval_report_ft.json 2>&1 | tail -25

## 9. Before / After comparison

Pretty-print the two reports side by side. Copy the resulting table into the README's evaluation section.

In [ ]:
import json
from pathlib import Path

mock = json.loads(Path('results/eval_report.json').read_text())
ft = json.loads(Path('results/eval_report_ft.json').read_text())

rows = []
for name, mock_m in mock['metrics'].items():
    ft_m = ft['metrics'].get(name, {})
    rows.append({
        'metric': name,
        'threshold': mock_m['threshold'],
        'mock': round(mock_m['score'], 4),
        'fine_tuned': round(ft_m.get('score', float('nan')), 4),
        'delta': round(ft_m.get('score', 0) - mock_m['score'], 4),
        'mock_pass': '✅' if mock_m['passed'] else '❌',
        'ft_pass': '✅' if ft_m.get('passed') else '❌',
    })

# Print as a markdown table the README can ingest directly
print('| Metric | Threshold | Mock | Fine-tuned | Δ | Mock | FT |')
print('|---|---|---|---|---|---|---|')
for r in rows:
    print(f"| {r['metric']} | {r['threshold']} | {r['mock']} | {r['fine_tuned']} | {r['delta']:+.4f} | {r['mock_pass']} | {r['ft_pass']} |")

print('\n--- raw fine-tuned report ---')
print(json.dumps(ft, indent=2)[:1200])

## 10. (Optional) Persist artifacts to Drive

Colab disks are ephemeral. To keep the merged model + report across sessions, mount Drive and copy. Skip if you're committing the report to git directly.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/npc-dialogue-engine
# !cp -r models/merged /content/drive/MyDrive/npc-dialogue-engine/
# !cp results/eval_report_ft.json /content/drive/MyDrive/npc-dialogue-engine/

## 11. Commit the report back to the repo

Run this from your laptop, not Colab — Colab can't push to your repo without a deploy key.

```bash
# On your local machine, after copying eval_report_ft.json over:
git add results/eval_report_ft.json
git commit -m "feat(eval): add fine-tuned model eval report (Qwen 2.5-3B + LoRA r=16)"
git push
```

Then update the README evaluation table with the markdown row from cell 9.